# 决策树（DT）

模式识别：在**已有特征**上按纯度（信息增益 / 基尼）选分裂，得到轴平行决策边界。  
不是「自动提取特征」——特征仍由人给定；树只是学怎么用这些特征做规则。

本笔记：二分类小表，手算一层最优分裂，对照 `DecisionTreeClassifier`。集成（随机森林等）一句带过。


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

# 特征: x0 是否晴天(1/0), x1 湿度(高1/低0)；标签: 出门(1)/不出(0)
# 刻意让 x0 更可分
X = np.array([
    [1, 0],
    [1, 1],
    [1, 0],
    [0, 1],
    [0, 1],
    [0, 0],
])
y = np.array([1, 1, 1, 0, 0, 0])
print("X,y:\n", np.column_stack([X, y]))


## 1. 基尼不纯度与信息增益（一层分裂）

基尼：\(G = 1 - \sum_c p_c^2\)。  
分裂后加权基尼越小越好（等价于基尼增益越大）。


In [ ]:
def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)

def best_split(X, y):
    n, d = X.shape
    parent = gini(y)
    best = None  # (gain, feature, threshold, left_mask)
    for j in range(d):
        for thr in np.unique(X[:, j]):
            left = X[:, j] <= thr
            right = ~left
            if left.sum() == 0 or right.sum() == 0:
                continue
            g = (left.sum() * gini(y[left]) + right.sum() * gini(y[right])) / n
            gain = parent - g
            if best is None or gain > best[0]:
                best = (gain, j, thr, left)
    return best

parent_g = gini(y)
gain, j, thr, left = best_split(X, y)
print(f"根节点基尼: {parent_g:.4f}")
print(f"最优分裂: 特征 x{j} <= {thr}, 基尼增益={gain:.4f}")
print("左子集标签:", y[left], "右子集标签:", y[~left])


## 2. sklearn 对照


In [ ]:
# max_depth=1：只看第一层分裂
tree = DecisionTreeClassifier(criterion="gini", max_depth=1, random_state=0)
tree.fit(X, y)
print(export_text(tree, feature_names=["x0", "x1"]))
print("feature_importances_:", tree.feature_importances_)
# 根节点特征应与手算一致
root_feat = tree.tree_.feature[0]
print("sklearn 根分裂特征索引:", root_feat, "一致?", root_feat == j)


## 3. 边界说明

- 树在给定特征上选分裂 ≠ 深度学习的自动提特征。  
- 随机森林 / GBDT：多棵树投票或串联，降低单树方差；本篇不展开。
